# Visits that cover tracts that cover visits

and 
### Visits that cover healpix'els that cover visits
### Tracts that cover visits when no data products exist yet. 

Context is we need to rerun  step1a and step1b on certain visits. What downstream tracts do they touch? and what downstream visits touch those tracts?

In [2]:
from lsst.daf.butler import Butler

In [3]:
butler = Butler("dp2_prep", collections="LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1")

In [3]:
# We have "isolated_star_association" with visits and tracts already. Using these woudl be very fast.

# if we haven't already run step1c, but have run step1a or step1b we could use 
# "preliminary_visit_image", "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1"

# and if we haven't run anything at all we could use:
#"raw" : 
# epoch1: LSSTCam/raw/DP2/20250424-20250702/DM-53733
# epoch2: LSSTCam/raw/DP2/20250703-20250921/DM-53733
# epoch3: LSSTCam/raw/DP2/20251024-20260106/DM-53733

### Tracts that cover a per-visit dataset (for which we've never aggregated into tracts yet)? 

For example, how would cm-service group step1c into groups of tracts if we have no idea what tracts are going to be covered yet? 

In [ ]:
# tracts that overlap "raw" in tagged collections
# runs out of memory in my 16GB notebook
with butler.query() as q:
    q = q.where(instrument="LSSTCam", skymap="lsst_cells_v2")
    q = q.where("tract.region OVERLAPS visit.region")
    q = q.join_dataset_search("raw", ["LSSTCam/raw/DP2/20250424-20250702/DM-53733",
                                      "LSSTCam/raw/DP2/20250703-20250921/DM-53733",
                                      "LSSTCam/raw/DP2/20251024-20260106/DM-53733"
                                     ])
    tracts = list(q.data_ids(["tract"]))

In [ ]:
len(tracts)

In [ ]:
# or tracts that overlap "preliminary_visit_image" in "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1"
# runs out of memory in a 16GB notebook
with butler.query() as q:
    q = q.where(instrument="LSSTCam", skymap="lsst_cells_v2")
    q = q.join_dataset_search("preliminary_visit_image", "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1")
    tracts = list(q.data_ids(["tract"]))

In [ ]:
len(tracts)

### Visits that cover tracts/healpix that cover visits

In [ ]:
# Which visits of refitPsfModel do we need to rerun if we've skipped day_obs=20260103 and rerun day_obs=20250702?

with butler.query() as q:
    q = q.where(instrument="LSSTCam", skymap="lsst_cells_v2")
    q = q.join_dataset_search("preliminary_visit_image", "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1")
    q = q.where("tract.region OVERLAPS visit.region")
    tracts = list(q.where("day_obs=20260103 OR day_obs=20250702").data_ids(["tract"]))
    visits = list(q.join_data_coordinates(tracts).data_ids(["visit"]))

# len(visits) returns 19883! which is 2/3 of all the visits
# len(tracts) returns 1694.
# This also expands beyond the 16GB of memory in my notebook

In [ ]:
# You can save the table of visits to use as input in quantum graph generation: 
import astropy.table
visitTable = astropy.table.Table(rows=[d.required_values for d in visits], names=list(visits[0].dimensions.required))
visitTable.write("visits.ecsv")

In [ ]:
# visitsB that cover tracts that cover visitsA when we DO have "isolated_star_association"
# "day_obs=20260103 OR day_obs=20250702"
# didn't get to the example using "isolated_star_association" etc..

In [ ]:
# Healpix for GBDES.  Need to rerun all healpix which touch a "single_visit_star"/"preliminary_visit_image" from
# "day_obs=20260103 OR day_obs=20250702 OR visit=2025050300588"

# Use fact that there is already one gbdesHealpix3AstrometricFitSkyWcsCatalog per healpix/visit combo created. 
# If 2025050300588 wasn't run yet, will it really end up in this list?

In [4]:
with butler.query() as q:
    q = q.where(instrument="LSSTCam", skymap="lsst_cells_v2")
    q = q.join_dataset_search("gbdesHealpix3AstrometricFitSkyWcsCatalog",
                              "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2")
    healpix3 = list(q.where("day_obs=20260103 OR day_obs=20250702 OR visit=2025050300588").data_ids(["healpix3", "band"])) # need band here
    dataIds  = list(q.join_data_coordinates(healpix3).data_ids(["healpix3", "visit"]))

len(dataIds)

18031

In [5]:
len(set(d["visit"] for d in dataIds))

8025

In [6]:
len(set(d["healpix3"] for d in dataIds))

121

In [ ]:
import astropy.table
gbdesTable = astropy.table.Table(rows=[d.required_values for d in dataIds], names=list(dataIds[0].dimensions.required))
gbdesTable.write("rerunGBDES2.ecsv")

In [ ]:
gbdesTable.sort("healpix3")

In [7]:
# gbdes runs per band, so are we sure that this is grabbing only the visits of the same band?
# 20260103 took data in only r,z,y, so we only want r,z,y, visits.

with butler.query() as q:
    q = q.where(instrument="LSSTCam", skymap="lsst_cells_v2")
    q = q.join_dataset_search("gbdesHealpix3AstrometricFitSkyWcsCatalog",
                              "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2")
    healpix3 = list(q.where("day_obs=20260103").data_ids(["healpix3", "band"]))  # NEED BAND HERE!
    dataIds  = list(q.join_data_coordinates(healpix3).data_ids(["healpix3", "visit"]))

In [8]:
# Yep, with band in the last where, we get only the healpix, visit, combos with the right band. 
set([d["band"] for d in dataIds])

{'r', 'y', 'z'}

In [9]:
# But it looks like we can't get visit=2025050300588 this way
# Not sure how to get the healpix,visit combos that visit=2025050300588 would have covered. 

with butler.query() as q:
    q = q.where(instrument="LSSTCam", skymap="lsst_cells_v2")
    q = q.join_dataset_search("gbdesHealpix3AstrometricFitSkyWcsCatalog",
                              "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2")
    healpix3 = list(q.where("visit=2025050300588").data_ids(["healpix3"]))
    dataIds  = list(q.join_data_coordinates(healpix3).data_ids(["healpix3", "visit"]))

InvalidQueryError: Cannot upload an empty data coordinate set.